# Dose-dependent morphokinetic endpoints — Met/Gab1

This notebook runs **13 morphokinetic endpoint analyses** sequentially,
using standalone scripts from `morphokinetic_endpoints_scripts.zip`.

Each section is independent — if you re-run only one section it will
reload its inputs from `OUTDIR`.

---

## USER INPUTS

Set the variables below and then run every cell top-to-bottom.

In [ ]:
# ── Paths ──
ZIP_PATH   = "/content/morphokinetic_endpoints_scripts.zip"
INPUT_PATH = "/content/data.csv"   # CSV or Parquet
OUTDIR     = "/content/out"

# ── Column names ──
EXPERIMENT_COL  = "Experiment"
PARENT_COL      = "Parent"
TIME_COL        = "Time"          # set None if data is already per-cell
CONDITION_COL   = "Condition"     # treatment / condition label
DOSE_COL        = "Dose"          # continuous proxy (optional)
DOSECATEGORY_COL= "DoseCategory"

# ── Protein-specific columns (optional) ──
METR_NORM_COL   = "METR_Norm"
GABY_NORM_COL   = "GABY_Norm"
METR_CAT_COL    = "METR_Category"
GABY_CAT_COL    = "GABY_Category"

# ── Clustering / PCA ──
CLUSTER_COL = "Cluster"
PC1_COL     = "PC1"
PC2_COL     = "PC2"
K           = 3

# ── Statistical parameters ──
N_PERM      = 500
N_BOOT      = 2000
REF_GROUP   = "control"   # must match a value in GROUP_COL

# ── Curve fitting / forecast ──
CURVE_FEATURE    = "Area"
FORECAST_FEATURE = "Area"
FORECAST_LAG     = 3
FORECAST_ALPHA   = 1.0

# ── Discriminability ──
DISCRIM_LABEL_COL = CONDITION_COL
OUTER_SPLITS      = 5

---
## SETUP

Install requirements, unzip scripts, and verify.

In [ ]:
!pip install -q numpy pandas scipy scikit-learn matplotlib statsmodels

import os, zipfile, shutil, sys

SCRIPTS_DIR = "/content/morphokinetic_endpoints"
if os.path.isdir(SCRIPTS_DIR):
    shutil.rmtree(SCRIPTS_DIR)

if os.path.exists(ZIP_PATH):
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall("/content")
    print(f"Unzipped {ZIP_PATH}")
else:
    print(f"⚠ ZIP not found at {ZIP_PATH} — using scripts already in {SCRIPTS_DIR}")

scripts = sorted([f for f in os.listdir(SCRIPTS_DIR) if f.endswith('.py')]) if os.path.isdir(SCRIPTS_DIR) else []
print(f"\nScripts found ({len(scripts)}):")
for s in scripts:
    print(f"  {s}")

import numpy, pandas, scipy, sklearn, matplotlib
print(f"\nnumpy={numpy.__version__}  pandas={pandas.__version__}  "
      f"scipy={scipy.__version__}  sklearn={sklearn.__version__}  "
      f"matplotlib={matplotlib.__version__}")

os.makedirs(OUTDIR, exist_ok=True)
print(f"Output dir: {OUTDIR}")

---
## DATA LOAD + SCHEMA CHECK

Load the input dataset and verify required columns.

In [ ]:
import pandas as pd, numpy as np

if INPUT_PATH.endswith(".parquet"):
    df_raw = pd.read_parquet(INPUT_PATH)
else:
    df_raw = pd.read_csv(INPUT_PATH)

print(f"Shape: {df_raw.shape}")
display(df_raw.head())

# ── Check required & optional columns ──
def _check(col, label, required=True):
    ok = col is not None and col in df_raw.columns
    tag = "✅" if ok else ("❌ REQUIRED" if required else "⚠ optional, will skip dependent steps")
    print(f"  {label:25s} ({str(col):20s}): {tag}")
    return ok

print("\nColumn check:")
HAS_EXP    = _check(EXPERIMENT_COL,  "Experiment")
HAS_PARENT = _check(PARENT_COL,      "Parent / Cell ID")
HAS_TIME   = _check(TIME_COL,        "Time",         required=False)
HAS_COND   = _check(CONDITION_COL,   "Condition",    required=False)
HAS_DOSE   = _check(DOSE_COL,        "Dose (cont.)", required=False)
HAS_DOSECAT= _check(DOSECATEGORY_COL,"DoseCategory", required=False)
HAS_METR_N = _check(METR_NORM_COL,   "METR_Norm",    required=False)
HAS_GABY_N = _check(GABY_NORM_COL,   "GABY_Norm",    required=False)
HAS_METR_C = _check(METR_CAT_COL,    "METR_Category",required=False)
HAS_GABY_C = _check(GABY_CAT_COL,    "GABY_Category",required=False)
HAS_CLUSTER= _check(CLUSTER_COL,     "Cluster",      required=False)

# Identify numeric feature columns
META = {EXPERIMENT_COL, PARENT_COL, TIME_COL, CONDITION_COL, DOSE_COL,
        DOSECATEGORY_COL, METR_NORM_COL, GABY_NORM_COL, METR_CAT_COL,
        GABY_CAT_COL, CLUSTER_COL, PC1_COL, PC2_COL,
        'Unnamed: 0','ID','x_Pos','y_Pos','dt','ds','unique_id','DoseCombo','DoseLabel'}
NUMERIC_FEATURES = [c for c in df_raw.columns
                    if c not in META and pd.api.types.is_numeric_dtype(df_raw[c])]
print(f"\nNumeric features detected: {len(NUMERIC_FEATURES)}")
print(NUMERIC_FEATURES[:20], '...' if len(NUMERIC_FEATURES) > 20 else '')

---
## Endpoint 1 — Continuous → DoseCategory (Neg / Pos / High)

**What it does:** Converts a continuous normalised dose column into three
ordered categories using fixed cutoffs (default −0.524 / 0.524) or
data-driven quantiles.

**Required columns:** At least one of `METR_NORM_COL`, `GABY_NORM_COL`, or `DOSE_COL`.

**Outputs:** `{OUTDIR}/table_with_dose_categories.csv`

In [ ]:
import subprocess, shlex

ep1_input = INPUT_PATH
ep1_ran = False

# ── METR ──
if HAS_METR_N:
    cmd = (f'python {SCRIPTS_DIR}/01_make_dose_categories.py '
           f'--input "{ep1_input}" --dose-col {METR_NORM_COL} '
           f'--method fixed --low-cut -0.524 --high-cut 0.524 '
           f'--out-col {METR_CAT_COL} --outdir "{OUTDIR}"')
    print(f"Running: {cmd}\n")
    !{cmd}
    ep1_input = os.path.join(OUTDIR, "table_with_dose_categories.csv")
    ep1_ran = True
else:
    print(f"⚠ {METR_NORM_COL} not found — skipping METR categorisation")

# ── GABY ──
if HAS_GABY_N:
    cmd = (f'python {SCRIPTS_DIR}/01_make_dose_categories.py '
           f'--input "{ep1_input}" --dose-col {GABY_NORM_COL} '
           f'--method fixed --low-cut -0.524 --high-cut 0.524 '
           f'--out-col {GABY_CAT_COL} --outdir "{OUTDIR}"')
    print(f"\nRunning: {cmd}\n")
    !{cmd}
    ep1_input = os.path.join(OUTDIR, "table_with_dose_categories.csv")
    ep1_ran = True
else:
    print(f"⚠ {GABY_NORM_COL} not found — skipping GABY categorisation")

# ── Generic DOSE_COL ──
if HAS_DOSE and DOSE_COL not in [METR_NORM_COL, GABY_NORM_COL]:
    cmd = (f'python {SCRIPTS_DIR}/01_make_dose_categories.py '
           f'--input "{ep1_input}" --dose-col {DOSE_COL} '
           f'--method fixed --low-cut -0.524 --high-cut 0.524 '
           f'--out-col {DOSECATEGORY_COL} --outdir "{OUTDIR}"')
    print(f"\nRunning: {cmd}\n")
    !{cmd}
    ep1_ran = True

if ep1_ran:
    _df1 = pd.read_csv(os.path.join(OUTDIR, "table_with_dose_categories.csv"))
    cat_cols = [c for c in _df1.columns if c.endswith('_Category')]
    for c in cat_cols:
        print(f"\n{c}:\n{_df1[c].value_counts(dropna=False)}")
else:
    print("⚠ No dose columns available — Endpoint 1 skipped entirely.")

---
## Endpoint 2 — DoseCombo (e.g. METR:High|GABY:Neg)

**What it does:** Combines per-protein dose category columns into a single
pipe-delimited combo label.

**Required columns:** `METR_Category` and `GABY_Category` (from Endpoint 1 or input).

**Outputs:** `{OUTDIR}/table_with_dose_combo.csv`

In [ ]:
# Use latest table from Endpoint 1 if available
ep2_input = os.path.join(OUTDIR, "table_with_dose_categories.csv")
if not os.path.exists(ep2_input):
    ep2_input = INPUT_PATH

_df2_check = pd.read_csv(ep2_input, nrows=5)
ep2_has_metr = METR_CAT_COL in _df2_check.columns
ep2_has_gaby = GABY_CAT_COL in _df2_check.columns

if ep2_has_metr and ep2_has_gaby:
    cmd = (f'python {SCRIPTS_DIR}/02_make_dose_combo.py '
           f'--input "{ep2_input}" '
           f'--cat-cols {METR_CAT_COL} {GABY_CAT_COL} '
           f'--outdir "{OUTDIR}"')
    print(f"Running: {cmd}\n")
    !{cmd}
elif ep2_has_metr or ep2_has_gaby:
    cat_col = METR_CAT_COL if ep2_has_metr else GABY_CAT_COL
    cmd = (f'python {SCRIPTS_DIR}/02_make_dose_combo.py '
           f'--input "{ep2_input}" '
           f'--cat-cols {cat_col} '
           f'--outdir "{OUTDIR}"')
    print(f"Running (single protein): {cmd}\n")
    !{cmd}
else:
    print(f"⚠ Neither {METR_CAT_COL} nor {GABY_CAT_COL} found — Endpoint 2 skipped.")

---
## Endpoint 3 — Track-level summary

**What it does:** If data is per-timepoint, aggregates numeric features
per cell track (mean, std, min, max, first, last) grouped by
(Experiment, Parent).

**Required columns:** `TIME_COL`, `EXPERIMENT_COL`, `PARENT_COL`.

**Outputs:** `{OUTDIR}/track_level_summary.csv`

In [ ]:
# Determine best available input (with dose info if possible)
ep3_input = INPUT_PATH
for candidate in ["table_with_dose_combo.csv", "table_with_dose_categories.csv"]:
    p = os.path.join(OUTDIR, candidate)
    if os.path.exists(p):
        ep3_input = p
        break

if HAS_TIME:
    # Determine extra categorical columns to carry forward
    keep = []
    _df3_check = pd.read_csv(ep3_input, nrows=5)
    for c in [CONDITION_COL, DOSECATEGORY_COL, METR_CAT_COL, GABY_CAT_COL, "DoseCombo"]:
        if c and c in _df3_check.columns:
            keep.append(c)
    keep_str = ' '.join(keep) if keep else ''
    cmd = (f'python {SCRIPTS_DIR}/03_track_level_summary.py '
           f'--input "{ep3_input}" '
           f'--exp-col {EXPERIMENT_COL} --parent-col {PARENT_COL} --time-col {TIME_COL} '
           f'{"--keep-cols " + keep_str if keep_str else ""} '
           f'--outdir "{OUTDIR}"')
    print(f"Running: {cmd}\n")
    !{cmd}
    _df3 = pd.read_csv(os.path.join(OUTDIR, "track_level_summary.csv"))
    print(f"\nTrack summary shape: {_df3.shape}")
    display(_df3.head())
else:
    print("⚠ TIME_COL not found — data assumed already per-cell. Endpoint 3 skipped.")

---
## Endpoint 4 — KMeans clustering + PCA

**What it does:** Standardises numeric features, runs KMeans(k=K), projects
to 2 PCs, saves cluster assignments and a PCA scatter plot.

**Required:** Numeric feature columns.

**Outputs:** `{OUTDIR}/table_with_clusters_and_pcs.csv`, `{OUTDIR}/pca_kmeans.png`

In [ ]:
# Prefer track-level summary if it exists
ep4_input = os.path.join(OUTDIR, "track_level_summary.csv")
if not os.path.exists(ep4_input):
    # Try dose-enriched table
    for candidate in ["table_with_dose_combo.csv", "table_with_dose_categories.csv"]:
        p = os.path.join(OUTDIR, candidate)
        if os.path.exists(p):
            ep4_input = p; break
    else:
        ep4_input = INPUT_PATH

cmd = (f'python {SCRIPTS_DIR}/04_cluster_kmeans_pca.py '
       f'--input "{ep4_input}" --k {K} --outdir "{OUTDIR}"')
print(f"Running: {cmd}\n")
!{cmd}

# Display PCA plot inline
from IPython.display import Image, display as ipy_display
pca_png = os.path.join(OUTDIR, "pca_kmeans.png")
if os.path.exists(pca_png):
    ipy_display(Image(pca_png, width=700))

---
## Endpoint 5 — Differential abundance of clusters

**What it does:** Chi-square test on the Cluster × Group contingency table,
plus Cramér's V and an optional permutation test blocked by Experiment.

**Required:** `Cluster`, a group column (DoseCombo > DoseCategory > Condition).

**Outputs:** `{OUTDIR}/cluster_contingency.csv`, `{OUTDIR}/cluster_diff_abundance_stats.csv`

In [ ]:
ep5_input = os.path.join(OUTDIR, "table_with_clusters_and_pcs.csv")
if not os.path.exists(ep5_input):
    print("⚠ Clustering output not found — run Endpoint 4 first."); raise SystemExit

_df5 = pd.read_csv(ep5_input, nrows=5)
# Pick best group column
GROUP_COL = None
for candidate in ["DoseCombo", DOSECATEGORY_COL, CONDITION_COL]:
    if candidate and candidate in _df5.columns:
        GROUP_COL = candidate; break

if GROUP_COL is None:
    print("⚠ No group column found (DoseCombo, DoseCategory, Condition) — Endpoint 5 skipped.")
else:
    exp_flag = f'--exp-col {EXPERIMENT_COL}' if EXPERIMENT_COL in _df5.columns else ''
    cmd = (f'python {SCRIPTS_DIR}/05_cluster_differential_abundance.py '
           f'--input "{ep5_input}" --group-col {GROUP_COL} '
           f'--cluster-col Cluster {exp_flag} --n-perm {N_PERM} '
           f'--outdir "{OUTDIR}"')
    print(f"Group column: {GROUP_COL}")
    print(f"Running: {cmd}\n")
    !{cmd}

---
## Endpoint 6 — State-transition rate per track

**What it does:** For time-resolved clustered data, computes the fraction
of consecutive timepoints where the cluster label changes.

**Required:** `TIME_COL`, `Cluster` in per-timepoint data.

**Outputs:** `{OUTDIR}/transition_rates.csv`, `{OUTDIR}/transition_rate_hist.png`

In [ ]:
if not HAS_TIME:
    print("⚠ TIME_COL not available — Endpoint 6 skipped (requires per-timepoint data).")
else:
    # We need per-timepoint data WITH cluster labels
    # Re-cluster the raw data if needed
    ep6_input = INPUT_PATH
    _df6_check = pd.read_csv(ep6_input, nrows=5)
    if "Cluster" not in _df6_check.columns:
        print("ℹ Cluster column not in raw data — assigning clusters from track-level results...")
        _tracks = pd.read_csv(os.path.join(OUTDIR, "table_with_clusters_and_pcs.csv"))
        _cluster_map = _tracks[[EXPERIMENT_COL, PARENT_COL, "Cluster"]].drop_duplicates()
        _df6 = pd.read_csv(ep6_input)
        _df6 = _df6.merge(_cluster_map, on=[EXPERIMENT_COL, PARENT_COL], how="left")
        ep6_tmp = os.path.join(OUTDIR, "_raw_with_clusters.csv")
        _df6.to_csv(ep6_tmp, index=False)
        ep6_input = ep6_tmp

    group_flag = ''
    if GROUP_COL:
        _df6_cols = pd.read_csv(ep6_input, nrows=5).columns
        if GROUP_COL in _df6_cols:
            group_flag = f'--group-col {GROUP_COL}'

    cmd = (f'python {SCRIPTS_DIR}/06_state_transition_rate.py '
           f'--input "{ep6_input}" '
           f'--exp-col {EXPERIMENT_COL} --parent-col {PARENT_COL} --time-col {TIME_COL} '
           f'--cluster-col Cluster {group_flag} --outdir "{OUTDIR}"')
    print(f"Running: {cmd}\n")
    !{cmd}

    from IPython.display import Image, display as ipy_display
    tr_png = os.path.join(OUTDIR, "transition_rate_hist.png")
    if os.path.exists(tr_png):
        ipy_display(Image(tr_png, width=600))

---
## Endpoint 7 — Distribution shift distances

**What it does:** Computes pairwise Wasserstein, KS, and Energy distances
between groups on a chosen value column (default: PC1).

**Required:** `PC1_COL`, a group column.

**Outputs:** `{OUTDIR}/distribution_shift_distances.csv`, `{OUTDIR}/distribution_shift_heatmap.png`

In [ ]:
ep7_input = os.path.join(OUTDIR, "table_with_clusters_and_pcs.csv")
if not os.path.exists(ep7_input):
    print("⚠ Clustering output not found — run Endpoint 4 first.")
elif GROUP_COL is None:
    print("⚠ No group column — Endpoint 7 skipped.")
else:
    cmd = (f'python {SCRIPTS_DIR}/07_distribution_shift_distances.py '
           f'--input "{ep7_input}" --group-col {GROUP_COL} '
           f'--value-col {PC1_COL} --outdir "{OUTDIR}"')
    print(f"Running: {cmd}\n")
    !{cmd}

    from IPython.display import Image, display as ipy_display
    hm_png = os.path.join(OUTDIR, "distribution_shift_heatmap.png")
    if os.path.exists(hm_png):
        ipy_display(Image(hm_png, width=600))

---
## Endpoint 8 — Dispersion / heterogeneity change

**What it does:** Levene's test (Brown-Forsythe) comparing within-group
variance of PC1 across groups. Also reports per-group variance and IQR.

**Required:** `PC1_COL`, a group column.

**Outputs:** `{OUTDIR}/dispersion_per_group.csv`, `{OUTDIR}/dispersion_stats.csv`

In [ ]:
ep8_input = os.path.join(OUTDIR, "table_with_clusters_and_pcs.csv")
if not os.path.exists(ep8_input):
    print("⚠ Clustering output not found — run Endpoint 4 first.")
elif GROUP_COL is None:
    print("⚠ No group column — Endpoint 8 skipped.")
else:
    cmd = (f'python {SCRIPTS_DIR}/08_dispersion_tests.py '
           f'--input "{ep8_input}" --group-col {GROUP_COL} '
           f'--value-col {PC1_COL} --outdir "{OUTDIR}"')
    print(f"Running: {cmd}\n")
    !{cmd}

---
## Endpoint 9 — Mean PC shift + bootstrap CI (experiment-block)

**What it does:** Computes per-group means of PC1 and PC2 with 95% confidence
intervals via experiment-block bootstrap.

**Required:** `PC1_COL`, `PC2_COL`, `EXPERIMENT_COL`, a group column.

**Outputs:** `{OUTDIR}/mean_pc_shift_bootstrap.csv`, `{OUTDIR}/mean_pc_shift_bootstrap.png`

In [ ]:
ep9_input = os.path.join(OUTDIR, "table_with_clusters_and_pcs.csv")
if not os.path.exists(ep9_input):
    print("⚠ Clustering output not found — run Endpoint 4 first.")
elif GROUP_COL is None:
    print("⚠ No group column — Endpoint 9 skipped.")
else:
    cmd = (f'python {SCRIPTS_DIR}/09_mean_pc_shift_bootstrap.py '
           f'--input "{ep9_input}" --group-col {GROUP_COL} '
           f'--exp-col {EXPERIMENT_COL} --pc1 {PC1_COL} --pc2 {PC2_COL} '
           f'--n-boot {N_BOOT} --outdir "{OUTDIR}"')
    print(f"Running: {cmd}\n")
    !{cmd}

    from IPython.display import Image, display as ipy_display
    bs_png = os.path.join(OUTDIR, "mean_pc_shift_bootstrap.png")
    if os.path.exists(bs_png):
        ipy_display(Image(bs_png, width=800))

---
## Endpoint 10 — Feature effect sizes + MWU + FDR vs REF_GROUP

**What it does:** For each numeric feature, computes Mann-Whitney U vs a
reference group, Cohen's d, and applies FDR (Benjamini-Hochberg).
Produces a volcano-style plot (effect size vs −log₁₀(q)).

**Required:** Numeric features, a group column, `REF_GROUP` must match a value.

**Outputs:** `{OUTDIR}/feature_effect_sizes.csv`, `{OUTDIR}/volcano_effect_sizes.png`

In [ ]:
ep10_input = os.path.join(OUTDIR, "table_with_clusters_and_pcs.csv")
if not os.path.exists(ep10_input):
    ep10_input = os.path.join(OUTDIR, "track_level_summary.csv")
if not os.path.exists(ep10_input):
    ep10_input = INPUT_PATH

if GROUP_COL is None:
    print("⚠ No group column — Endpoint 10 skipped.")
else:
    _df10 = pd.read_csv(ep10_input, nrows=100)
    if GROUP_COL not in _df10.columns:
        print(f"⚠ {GROUP_COL} not in {ep10_input} — Endpoint 10 skipped.")
    elif REF_GROUP not in _df10[GROUP_COL].unique():
        available = list(_df10[GROUP_COL].dropna().unique())
        print(f"⚠ REF_GROUP='{REF_GROUP}' not found in {GROUP_COL}. Available: {available}")
        print("   Update REF_GROUP in the USER INPUTS cell and rerun.")
    else:
        cmd = (f'python {SCRIPTS_DIR}/10_feature_effect_sizes_fdr.py '
               f'--input "{ep10_input}" --group-col {GROUP_COL} '
               f'--ref-group "{REF_GROUP}" --outdir "{OUTDIR}"')
        print(f"Running: {cmd}\n")
        !{cmd}

        from IPython.display import Image, display as ipy_display
        vol_png = os.path.join(OUTDIR, "volcano_effect_sizes.png")
        if os.path.exists(vol_png):
            ipy_display(Image(vol_png, width=800))

---
## Endpoint 11 — 10-model curve fitting (AIC selection)

**What it does:** Fits 10 parametric models (linear, quadratic, cubic,
exp-growth, exp-decay, logistic, gompertz, power, log, plateau) to
`CURVE_FEATURE` per track and selects the best by AIC.

**Required:** `TIME_COL`, `CURVE_FEATURE`.

**Outputs:** `{OUTDIR}/curve_fitting_results.csv`, `{OUTDIR}/curve_fit_model_dist.png`

In [ ]:
if not HAS_TIME:
    print("⚠ TIME_COL not available — Endpoint 11 skipped.")
elif CURVE_FEATURE not in df_raw.columns:
    print(f"⚠ CURVE_FEATURE='{CURVE_FEATURE}' not in data — Endpoint 11 skipped.")
else:
    ep11_input = INPUT_PATH
    cmd = (f'python {SCRIPTS_DIR}/11_curve_fitting_10_models.py '
           f'--input "{ep11_input}" '
           f'--exp-col {EXPERIMENT_COL} --parent-col {PARENT_COL} --time-col {TIME_COL} '
           f'--feature {CURVE_FEATURE} --outdir "{OUTDIR}"')
    print(f"Running: {cmd}\n")
    !{cmd}

    from IPython.display import Image, display as ipy_display
    cf_png = os.path.join(OUTDIR, "curve_fit_model_dist.png")
    if os.path.exists(cf_png):
        ipy_display(Image(cf_png, width=700))

    # Show best model by Condition if available
    _cf = pd.read_csv(os.path.join(OUTDIR, "curve_fitting_results.csv"))
    if CONDITION_COL:
        _cond_map = df_raw.groupby([EXPERIMENT_COL, PARENT_COL])[CONDITION_COL].first().reset_index()
        _cf = _cf.merge(_cond_map, on=[EXPERIMENT_COL, PARENT_COL], how="left")
        if CONDITION_COL in _cf.columns:
            print(f"\nBest model by {CONDITION_COL}:")
            print(pd.crosstab(_cf[CONDITION_COL], _cf['best_model']))

---
## Endpoint 12 — Forecast predictability

**What it does:** Per-track Ridge regression to predict next value of
`FORECAST_FEATURE` from the previous `FORECAST_LAG` values. Reports MAE.

**Required:** `TIME_COL`, `FORECAST_FEATURE`.

**Outputs:** `{OUTDIR}/forecast_mae.csv`, `{OUTDIR}/forecast_mae_boxplot.png`

In [ ]:
if not HAS_TIME:
    print("⚠ TIME_COL not available — Endpoint 12 skipped.")
elif FORECAST_FEATURE not in df_raw.columns:
    print(f"⚠ FORECAST_FEATURE='{FORECAST_FEATURE}' not in data — Endpoint 12 skipped.")
else:
    ep12_input = INPUT_PATH
    group_flag = ''
    if GROUP_COL and GROUP_COL in df_raw.columns:
        group_flag = f'--group-col {GROUP_COL}'
    elif CONDITION_COL and CONDITION_COL in df_raw.columns:
        group_flag = f'--group-col {CONDITION_COL}'

    cmd = (f'python {SCRIPTS_DIR}/12_forecast_predictability.py '
           f'--input "{ep12_input}" '
           f'--exp-col {EXPERIMENT_COL} --parent-col {PARENT_COL} --time-col {TIME_COL} '
           f'--feature {FORECAST_FEATURE} --lag {FORECAST_LAG} --alpha {FORECAST_ALPHA} '
           f'{group_flag} --outdir "{OUTDIR}"')
    print(f"Running: {cmd}\n")
    !{cmd}

    from IPython.display import Image, display as ipy_display
    mae_png = os.path.join(OUTDIR, "forecast_mae_boxplot.png")
    if os.path.exists(mae_png):
        ipy_display(Image(mae_png, width=600))

---
## Endpoint 13 — Discriminability (nested GroupKFold, blocked by Experiment)

**What it does:** L1-penalised logistic regression to predict a label column
from numeric features, using GroupKFold CV with groups = Experiment.
Reports balanced accuracy and macro F1 per fold.

**Required:** `DISCRIM_LABEL_COL`, `EXPERIMENT_COL`, numeric features.

**Outputs:** `{OUTDIR}/discriminability_cv.csv`, `{OUTDIR}/discriminability_folds.png`

In [ ]:
# Use best available track-level table
ep13_input = os.path.join(OUTDIR, "table_with_clusters_and_pcs.csv")
if not os.path.exists(ep13_input):
    ep13_input = os.path.join(OUTDIR, "track_level_summary.csv")
if not os.path.exists(ep13_input):
    # Aggregate raw data on the fly
    print("ℹ No track-level file found — aggregating raw data per cell...")
    _df13 = df_raw.groupby([EXPERIMENT_COL, PARENT_COL])[NUMERIC_FEATURES].mean().reset_index()
    if DISCRIM_LABEL_COL and DISCRIM_LABEL_COL in df_raw.columns:
        _lbl = df_raw.groupby([EXPERIMENT_COL, PARENT_COL])[DISCRIM_LABEL_COL].first().reset_index()
        _df13 = _df13.merge(_lbl, on=[EXPERIMENT_COL, PARENT_COL], how="left")
    ep13_input = os.path.join(OUTDIR, "_agg_for_discrim.csv")
    _df13.to_csv(ep13_input, index=False)

_df13_check = pd.read_csv(ep13_input, nrows=5)
if DISCRIM_LABEL_COL not in _df13_check.columns:
    print(f"⚠ DISCRIM_LABEL_COL='{DISCRIM_LABEL_COL}' not found — Endpoint 13 skipped.")
else:
    cmd = (f'python {SCRIPTS_DIR}/13_nested_groupcv_discriminability.py '
           f'--input "{ep13_input}" --label-col {DISCRIM_LABEL_COL} '
           f'--exp-col {EXPERIMENT_COL} --outer-splits {OUTER_SPLITS} '
           f'--outdir "{OUTDIR}"')
    print(f"Running: {cmd}\n")
    !{cmd}

    from IPython.display import Image, display as ipy_display
    disc_png = os.path.join(OUTDIR, "discriminability_folds.png")
    if os.path.exists(disc_png):
        ipy_display(Image(disc_png, width=700))

---
## RESULTS SUMMARY

Loads all produced outputs and prints a compact report.

In [ ]:
import os, pandas as pd

print("=" * 70)
print("RESULTS SUMMARY")
print("=" * 70)

# ── File inventory ──
produced = sorted(os.listdir(OUTDIR))
# Map endpoints to output files
endpoint_files = {
    "EP01 DoseCategory":           [f for f in produced if 'dose_categories' in f],
    "EP02 DoseCombo":              [f for f in produced if 'dose_combo' in f],
    "EP03 Track Summary":          [f for f in produced if 'track_level' in f],
    "EP04 KMeans + PCA":           [f for f in produced if 'clusters_and_pcs' in f or 'pca_kmeans' in f],
    "EP05 Diff Abundance":         [f for f in produced if 'contingency' in f or 'diff_abundance' in f],
    "EP06 Transition Rate":        [f for f in produced if 'transition' in f],
    "EP07 Distribution Shift":     [f for f in produced if 'distribution_shift' in f],
    "EP08 Dispersion":             [f for f in produced if 'dispersion' in f],
    "EP09 Bootstrap PC Shift":     [f for f in produced if 'bootstrap' in f],
    "EP10 Effect Sizes":           [f for f in produced if 'effect_size' in f or 'volcano' in f],
    "EP11 Curve Fitting":          [f for f in produced if 'curve_fit' in f],
    "EP12 Forecast":               [f for f in produced if 'forecast' in f],
    "EP13 Discriminability":       [f for f in produced if 'discriminability' in f],
}

print("\n── Output files per endpoint ──")
for ep, files in endpoint_files.items():
    status = ", ".join(files) if files else "(not produced)"
    print(f"  {ep:30s}: {status}")

# ── Key statistics ──
print("\n── Key statistics ──")

# Diff abundance
p = os.path.join(OUTDIR, "cluster_diff_abundance_stats.csv")
if os.path.exists(p):
    s = pd.read_csv(p)
    print(f"  Cluster diff abundance: chi2={s['chi2'].iloc[0]:.2f}, "
          f"p={s['p_chi2'].iloc[0]:.2e}, Cramer's V={s['cramers_v'].iloc[0]:.4f}"
          f"{', perm_p=' + f'{s.perm_p.iloc[0]:.4f}' if not pd.isna(s['perm_p'].iloc[0]) else ''}")

# Top distribution shifts
p = os.path.join(OUTDIR, "distribution_shift_distances.csv")
if os.path.exists(p):
    d = pd.read_csv(p).sort_values("Wasserstein", ascending=False)
    print(f"\n  Top 10 distribution shifts (Wasserstein on {PC1_COL}):")
    print(d.head(10).to_string(index=False))

# Dispersion
p = os.path.join(OUTDIR, "dispersion_stats.csv")
if os.path.exists(p):
    s = pd.read_csv(p)
    print(f"\n  Dispersion (Levene): stat={s['levene_stat'].iloc[0]:.4f}, p={s['levene_p'].iloc[0]:.4e}")

# Discriminability
p = os.path.join(OUTDIR, "discriminability_cv.csv")
if os.path.exists(p):
    d = pd.read_csv(p)
    print(f"\n  Discriminability: balanced_acc={d['balanced_acc'].mean():.3f} ± {d['balanced_acc'].std():.3f}, "
          f"macro_f1={d['macro_f1'].mean():.3f} ± {d['macro_f1'].std():.3f}")

print(f"\n  Total files in {OUTDIR}: {len(produced)}")
print("\n" + "=" * 70)
print("Done.")